<h1> Pipeline Assistente Autônomo </h1>
Objetivo: Crie todo o pipeline do assistente autônomo, ou seja, integra a RAG ao LLM via functions call. </br>
Entrada: Base de dados vetorizada (embeddings) </br>
Saída: Funcionalidade de assistente autônomo. Posteriormente uma interface gráfica será construída e terá também o módulo de avaliação.</br> 

<h3> Importação de pacotes e bibliotecas </h3>

In [1]:
import sys
from pathlib import Path

RAIZ_PROJETO = Path.cwd().parent

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

In [16]:
import pandas as pd
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from google import genai

c:\Users\roger\.conda\envs\itau\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Lista de imports novos
import json
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import pandas as pd
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

<h3> Definição de Métodos e Funções </h3>

In [5]:
def ler_base_por_csv(caminho_arquivo):
    """
    Lê um arquivo CSV e retorna um DataFrame do Pandas.
    """
    try:
        base_dataframe = pd.read_csv(caminho_arquivo)
        print(f"Arquivo lido com sucesso")
        return base_dataframe
    except Exception as excecao:
        print(f"Erro ao ler o arquivo: {excecao}")
        return None

Definição das ferramentas(tools) que o AGENTE AUTÔNOMO irá usar

<h3> Definição do Escopo Principal(Main) </h3>

Configurando PROXY

In [2]:
# config/network.py
import socket
if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo


def configurar_ipv4():
    if getattr(socket, "_ipv4_configurado", False):
        return

    original_getaddrinfo = socket._original_getaddrinfo

    def getaddrinfo_ipv4(*args, **kwargs):
        resultados = original_getaddrinfo(*args, **kwargs)

        resultados_ipv4 = [
            resultado
            for resultado in resultados
            if resultado[0] == socket.AF_INET
        ]

        return resultados_ipv4

    socket.getaddrinfo = getaddrinfo_ipv4
    socket._ipv4_configurado = True

configurar_ipv4()
print("Python configurado para priorizar IPv4.")

Python configurado para priorizar IPv4.


In [ ]:
# app.py teremos isso
#from config.network import configurar_ipv4
#configurar_ipv4()

In [ ]:
# Consulta a dados estruturados do CLIENTE é a tabela de clientes_sem_nulos sem as interações.

<h3> Importante: rascunhos </h3>

As células represetam testes isolados de ferramentas ou solução de problemas pontuais, não use como referência de evidência técnicas. Os códigos oficiais do assistente já estão no formato de script dentro da pasta do projeto.

<h4> 6. Teste com LANG CHAIN. Voltaremos posteriromente </h4>

Carregando a base de clientes estruturada

In [6]:
# Carregando a base
caminho_info_cadastrais = "../bases_tratadas/clientes_sem_nulos.csv"
df_clientes = ler_base_por_csv(caminho_info_cadastrais)

Arquivo lido com sucesso


In [13]:
resultado = consultar_cliente.invoke({
    "id_cliente": "PJ0004"
})

print(resultado)

{'sucesso': True, 'id_cliente': 'PJ0004', 'dados': {'id_cliente': 'PJ0004', 'data_referencia': '2026-01-28', 'setor': 'Tecnologia', 'porte': 'ME', 'uf': 'RS', 'tempo_relacionamento_meses': 153, 'faturamento_mensal_estimado': 32413.94, 'saldo_devedor': 8590.95, 'dias_atraso': 45, 'qtd_contratos_ativos': 1, 'qtd_parcelas_vencidas': 1, 'limite_credito': 21274.16, 'utilizacao_limite_pct': 0.3296, 'qtd_contatos_ult_30d': 1, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'portal', 'risco_setorial': 'alto', 'score_regularizacao_legado': 818, 'score_cobranca_atualizado_30d': 881, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 1, 'dia_referencia': 28, 'ano_mes_referencia': '2026-01', 'nome_mes_referencia': 'Janeiro'}}


Inicializando o GEMINI

In [16]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

Registrando a tool

In [17]:
llm_com_tools = llm.bind_tools([
    consultar_cliente
])

Testando a Ferramenta

In [24]:
id_teste = "PJ0005"

mensagem = llm_com_tools.invoke(
    f"Consulte o cliente com id {id_teste} e me informe o setor dele somente"
    # Consulte o cliente com id PJ0004 e me informe o setor dele.
)

In [27]:
print(mensagem)

content=[] additional_kwargs={'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_278527': 'EnEKbwERTTIPUyNPUVJ+fzHBlGG/yt/JKwP3WNx6LuYlgdFscfu2JNhnkA5KtHSpGcbWKHL3UzcB8oRDB18zRwZt3Qy3NN/HJgAnQsLk3IcBoWtfOMBNyN91f0BL761Ayp/7LjE5eCXPlFyr+drMnOd7PA=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a025c5-7183-7143-8a75-d6f27b959297-0' tool_calls=[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_278527', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 143, 'output_tokens': 23, 'total_tokens': 166, 'input_token_details': {'cache_read': 0}}


In [28]:
print(mensagem.tool_calls)

[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_278527', 'type': 'tool_call'}]


In [29]:
mensagem = llm_com_tools.invoke(
    f"Consulte o cliente com id {id_teste} e me informe o setor dele somente"
)

In [30]:
print("TOOL CALL:")
print(mensagem.tool_calls)

TOOL CALL:
[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_109018', 'type': 'tool_call'}]


In [ ]:
# teste - Chama explicito
tool_call = mensagem.tool_calls[0]

In [35]:
resultado_tool = consultar_cliente.invoke(
    tool_call["args"]
)

print(resultado_tool)

{'sucesso': True, 'id_cliente': 'PJ0005', 'dados': {'id_cliente': 'PJ0005', 'data_referencia': '2026-02-28', 'setor': 'Comércio', 'porte': 'ME', 'uf': 'SC', 'tempo_relacionamento_meses': 208, 'faturamento_mensal_estimado': 96854.12, 'saldo_devedor': 19745.94, 'dias_atraso': 43, 'qtd_contratos_ativos': 2, 'qtd_parcelas_vencidas': 1, 'limite_credito': 83140.62, 'utilizacao_limite_pct': 0.1913, 'qtd_contatos_ult_30d': 4, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'whatsapp', 'risco_setorial': 'baixo', 'score_regularizacao_legado': 527, 'score_cobranca_atualizado_30d': 909, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 2, 'dia_referencia': 28, 'ano_mes_referencia': '2026-02', 'nome_mes_referencia': 'Fevereiro'}}


In [36]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(resultado_tool),
    tool_call_id=tool_call["id"]
)

In [37]:
tool_message

ToolMessage(content="{'sucesso': True, 'id_cliente': 'PJ0005', 'dados': {'id_cliente': 'PJ0005', 'data_referencia': '2026-02-28', 'setor': 'Comércio', 'porte': 'ME', 'uf': 'SC', 'tempo_relacionamento_meses': 208, 'faturamento_mensal_estimado': 96854.12, 'saldo_devedor': 19745.94, 'dias_atraso': 43, 'qtd_contratos_ativos': 2, 'qtd_parcelas_vencidas': 1, 'limite_credito': 83140.62, 'utilizacao_limite_pct': 0.1913, 'qtd_contatos_ult_30d': 4, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'whatsapp', 'risco_setorial': 'baixo', 'score_regularizacao_legado': 527, 'score_cobranca_atualizado_30d': 909, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 2, 'dia_referencia': 28, 'ano_mes_referencia': '2026-02', 'nome_mes_referencia': 'Fevereiro'}}", tool_call_id='call_109018')

In [38]:
mensagens = [
    {
        "role": "user",
        "content": f"Consulte o cliente com id {id_teste} e me informe o setor dele somente"
    },
    mensagem,
    tool_message
]

In [39]:
resposta_final = llm_com_tools.invoke(mensagens)

In [40]:
print(resposta_final.content)

[{'type': 'text', 'text': 'O setor do cliente PJ0005 é Comércio.', 'extras': {'signature': 'EnEKbwERTTIP9ptEs9zPvH70pez9sLXVFU2zyhtWXqcc2g+G7E/8XUS0yKGagl3ctufp61DI07fr040OLDEe3lIW/kzbVQp75FlKq+gOs/PVxThXoYn4CffsFEXp02x4aONEuCoid8WgBd/k2UeozuBGtg=='}}]


Teste Final

In [41]:
from langchain_core.messages import ToolMessage

id_teste = "PJ0005"

# 1. Gemini decide qual Tool usar
mensagem = llm_com_tools.invoke(
    f"Consulte o cliente com id {id_teste} e me informe o setor dele somente"
)

print("Tool chamada:")
print(mensagem.tool_calls)


# 2. Executa a Tool
tool_call = mensagem.tool_calls[0]

resultado_tool = consultar_cliente.invoke(
    tool_call["args"]
)

print("\nResultado da Tool:")
print(resultado_tool)


# 3. Cria mensagem com o resultado da Tool
tool_message = ToolMessage(
    content=str(resultado_tool),
    tool_call_id=tool_call["id"]
)


# 4. Envia novamente para o Gemini
resposta_final = llm_com_tools.invoke([
    {
        "role": "user",
        "content": f"Consulte o cliente com id {id_teste} e me informe o setor dele somente"
    },
    mensagem,
    tool_message
])


# 5. Resposta final
print("\nResposta final:")
print(resposta_final.content)

Tool chamada:
[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_536019', 'type': 'tool_call'}]

Resultado da Tool:
{'sucesso': True, 'id_cliente': 'PJ0005', 'dados': {'id_cliente': 'PJ0005', 'data_referencia': '2026-02-28', 'setor': 'Comércio', 'porte': 'ME', 'uf': 'SC', 'tempo_relacionamento_meses': 208, 'faturamento_mensal_estimado': 96854.12, 'saldo_devedor': 19745.94, 'dias_atraso': 43, 'qtd_contratos_ativos': 2, 'qtd_parcelas_vencidas': 1, 'limite_credito': 83140.62, 'utilizacao_limite_pct': 0.1913, 'qtd_contatos_ult_30d': 4, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'whatsapp', 'risco_setorial': 'baixo', 'score_regularizacao_legado': 527, 'score_cobranca_atualizado_30d': 909, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 2, 'dia_referencia': 28, 'ano_mes_referencia': '2026-02', 'nome_mes_referencia': 'Fevereiro'}}

Resposta final:
[{'type': 'text', 'text': 'O setor do cliente PJ0005 é Comércio.'

Definição de Escopo NOVO E LIMPO 

In [31]:
# 1. Definição das ferramentas
tools = {
    "consultar_cliente": consultar_cliente,
    "consultar_documentos": consultar_documentos
}

In [10]:
# 2. Carregando a base para ser usada
caminho_info_cadastrais = "../bases_tratadas/clientes_sem_nulos.csv"
df_clientes = ler_base_por_csv(caminho_info_cadastrais)

Arquivo lido com sucesso


In [11]:
# 3. Credenciais ...
os.environ['GEMINI_API_KEY'] = 'AQ.Ab8RN6KG8l1nYtGTG_jBfUP-6dRixPbreJZcbcNWBlHwzx5b5Q'
os.environ['GEMINI_API_KEY']

'AQ.Ab8RN6KG8l1nYtGTG_jBfUP-6dRixPbreJZcbcNWBlHwzx5b5Q'

In [12]:
# 3. Criando a LLM 
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

In [32]:
# 4. Registrando a Tool
llm_com_tools = llm.bind_tools(
    list(tools.values())
)

In [14]:
# 5. Usando a LLM
resposta = executar_agente(
    pergunta="Consulte o cliente PJ0005 e informe o setor.",
    llm=llm_com_tools,
    tools=tools
)

print(resposta.content)

[{'type': 'text', 'text': 'O setor do cliente PJ0005 é Comércio.', 'extras': {'signature': 'EnEKbwERTTIPTrNmB+WQ/lhBS7zN/3pD5A2USXXig0mLUKGhS9sytSAP6pwgGNQct6SigkorcEifc1/TwoGJsqKHt8rV+6AB2T7hF9eDQaPANcAAg9YBJdBRS7oyzgrrjvcSDjM0Z9A/EouRxWMjaXLrWQ=='}}]


RAG

In [22]:
%%time
modelo_embedding = carregar_modelo_embedding()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1929.56it/s]


CPU times: total: 3.7 s
Wall time: 19 s


In [23]:
client, collection = carregar_colecao_rag(
    caminho_db="./chroma_db",
    nome_colecao="documentos"
)

Coleção 'documentos' carregada com sucesso.
Quantidade de documentos: 5


In [24]:
resultados = buscar_documentos(
    consulta="Quais são as faixas operacionais de atraso?",
    modelo_embedding=modelo_embedding,
    colecao=collection,
    n_resultados=5
)

In [25]:
resultados

[{'id_documento': '1',
  'texto': 'CRITÉRIOS PARA OFERTAS DE RENEGOCIAÇÃO PJ\nCódigo: COR-PJ-014 | Versão: 3.5 | Vigência fictícia: 01/07/2026\n\n1. PREMISSAS\nAs condições abaixo são fictícias. A elegibilidade deve ser calculada com dados atualizados. Recomendação não equivale a aprovação e nenhuma condição deve ser prometida antes da validação no motor de ofertas.\n\n2. AJUSTE DE VENCIMENTO\nElegível para atraso entre 1 e 30 dias, no máximo uma ocorrência nos últimos 12 meses e ausência de promessa vencida. Não há desconto de principal. Requer confirmação do fluxo de caixa esperado.\n\n3. PARCELAMENTO PADRONIZADO\n3.1. Atraso entre 16 e 90 dias.\n3.2. Saldo devedor até R$ 500.000.\n3.3. Entrada mínima de 10% do saldo consolidado.\n3.4. Prazo máximo de 24 meses.\n3.5. Cliente com até duas renegociações nos últimos 12 meses.\n3.6. Acima de R$ 250.000 ou acima de 60 dias de atraso, exigir validação de especialista.\n\n4. OFERTA CUSTOMIZADA\nPode ser analisada quando houver evidência de 

In [26]:
for resultado in resultados:
    print("=" * 80)
    print("ID:", resultado["id_documento"])
    print("Score:", resultado["score_similaridade"])
    print("Texto:", resultado["texto"][:500])

ID: 1
Score: 69.4401657210564
Texto: CRITÉRIOS PARA OFERTAS DE RENEGOCIAÇÃO PJ
Código: COR-PJ-014 | Versão: 3.5 | Vigência fictícia: 01/07/2026

1. PREMISSAS
As condições abaixo são fictícias. A elegibilidade deve ser calculada com dados atualizados. Recomendação não equivale a aprovação e nenhuma condição deve ser prometida antes da validação no motor de ofertas.

2. AJUSTE DE VENCIMENTO
Elegível para atraso entre 1 e 30 dias, no máximo uma ocorrência nos últimos 12 meses e ausência de promessa vencida. Não há desconto de principa
ID: 5
Score: 66.60124937424965
Texto: POLÍTICA DE RECUPERAÇÃO DE CRÉDITO PJ
Código: PRC-PJ-017 | Versão: 3.2 | Vigência fictícia: 01/07/2026
Proprietário: Diretoria de Recuperação PJ

1. OBJETIVO
Estabelecer princípios e limites para priorização, contato, negociação e encaminhamento de clientes Pessoa Jurídica em atraso. Esta política é fictícia e válida apenas no case técnico.

2. PRINCÍPIOS
2.1. Toda recomendação deve considerar capacidade de pagamento, es

In [29]:
resultado = consultar_documentos.invoke(
    {
        "consulta": "Quais são as faixas operacionais de atraso?"
    }
)

In [30]:
resultado

{'sucesso': True,
 'quantidade_resultados': 5,
 'documentos': [{'id_documento': '1',
   'texto': 'CRITÉRIOS PARA OFERTAS DE RENEGOCIAÇÃO PJ\nCódigo: COR-PJ-014 | Versão: 3.5 | Vigência fictícia: 01/07/2026\n\n1. PREMISSAS\nAs condições abaixo são fictícias. A elegibilidade deve ser calculada com dados atualizados. Recomendação não equivale a aprovação e nenhuma condição deve ser prometida antes da validação no motor de ofertas.\n\n2. AJUSTE DE VENCIMENTO\nElegível para atraso entre 1 e 30 dias, no máximo uma ocorrência nos últimos 12 meses e ausência de promessa vencida. Não há desconto de principal. Requer confirmação do fluxo de caixa esperado.\n\n3. PARCELAMENTO PADRONIZADO\n3.1. Atraso entre 16 e 90 dias.\n3.2. Saldo devedor até R$ 500.000.\n3.3. Entrada mínima de 10% do saldo consolidado.\n3.4. Prazo máximo de 24 meses.\n3.5. Cliente com até duas renegociações nos últimos 12 meses.\n3.6. Acima de R$ 250.000 ou acima de 60 dias de atraso, exigir validação de especialista.\n\n4. OFE

In [ ]:
# teste isolado
resposta = executar_agente(
    pergunta=(
        "Quais são as faixas operacionais "
        "de atraso previstas na Política de "
        "Recuperação de Crédito PJ?"
    ),
    llm=llm_com_tools,
    tools=tools
)

print(resposta.content)

In [3]:
from assistente.llm import carregar_llm

llm = carregar_llm()

2026-08-21 21:26:38.289 | INFO     | assistente.llm:carregar_llm:26 - Inicializando LLM Gemini: gemini-3.1-flash-lite
2026-08-21 21:26:38.457 | INFO     | assistente.llm:carregar_llm:34 - LLM Gemini inicializada com sucesso.


In [3]:
resposta = llm.invoke(
    "Responda apenas: OK"
)

print(resposta.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'OK', 'extras': {'signature': 'EnEKbwERTTIPKZ/0Yo6O7LtZ+3f+8mq8MTfXiVRV6NlNcnJCEyU+01RK7se7QLEL8zQoe1yb3jHr3geQMC8cxmAGTmTRijsVSl3ES0438O9ldGSyi63kaGBVcnKJpNGWwzc09s/IF9swMy/RMTmR1autig=='}}]


In [4]:
from assistente.tools import consultar_cliente

In [5]:
resultado = consultar_cliente.invoke(
    {"id_cliente": "PJ0005"}
)

print(resultado)

2026-08-21 20:53:28.695 | INFO     | assistente.tools:consultar_cliente:63 - Consultando cliente: PJ0005
2026-08-21 20:53:28.696 | INFO     | assistente.tools:carregar_clientes:26 - Carregando base estruturada de clientes.
2026-08-21 20:53:28.840 | INFO     | assistente.tools:carregar_clientes:30 - Base de clientes carregada com sucesso. Registros: 800
2026-08-21 20:53:28.853 | INFO     | assistente.tools:consultar_cliente:86 - Cliente PJ0005 encontrado com sucesso.


{'sucesso': True, 'id_cliente': 'PJ0005', 'dados': {'id_cliente': 'PJ0005', 'data_referencia': '2026-02-28', 'setor': 'Comércio', 'porte': 'ME', 'uf': 'SC', 'tempo_relacionamento_meses': 208, 'faturamento_mensal_estimado': 96854.12, 'saldo_devedor': 19745.94, 'dias_atraso': 43, 'qtd_contratos_ativos': 2, 'qtd_parcelas_vencidas': 1, 'limite_credito': 83140.62, 'utilizacao_limite_pct': 0.1913, 'qtd_contatos_ult_30d': 4, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'whatsapp', 'risco_setorial': 'baixo', 'score_regularizacao_legado': 527, 'score_cobranca_atualizado_30d': 909, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 2, 'dia_referencia': 28, 'ano_mes_referencia': '2026-02', 'nome_mes_referencia': 'Fevereiro'}}


Teste 2

In [5]:
from assistente.llm import (
    carregar_llm,
    carregar_llm_com_tools
)

from assistente.tools import consultar_cliente

2026-08-21 22:21:49.060 | INFO     | assistente.llm:carregar_llm:26 - Inicializando LLM Gemini: gemini-3.1-flash-lite
2026-08-21 22:21:49.246 | INFO     | assistente.llm:carregar_llm:34 - LLM Gemini inicializada com sucesso.


In [6]:
llm = carregar_llm()

2026-08-21 22:21:51.800 | INFO     | assistente.llm:carregar_llm:26 - Inicializando LLM Gemini: gemini-3.1-flash-lite
2026-08-21 22:21:51.861 | INFO     | assistente.llm:carregar_llm:34 - LLM Gemini inicializada com sucesso.


In [7]:
tools = [
    consultar_cliente
]

In [8]:
llm_com_tools = carregar_llm_com_tools(
    llm=llm,
    tools=tools
)

2026-08-21 22:22:02.759 | INFO     | assistente.llm:carregar_llm_com_tools:50 - Configurando Tool Calling com 1 ferramenta(s).
2026-08-21 22:22:02.767 | INFO     | assistente.llm:carregar_llm_com_tools:57 - LLM configurada com Tool Calling com sucesso.


In [ ]:
mensagem = llm_com_tools.invoke(
    "Consulte o cliente com id PJ0005."
)

print(mensagem)

In [7]:
import time

inicio = time.time()

mensagem = llm_com_tools.invoke(
    "Consulte o cliente com id PJ0005."
)

fim = time.time()

print(mensagem)
print(f"Tempo: {fim - inicio:.2f} segundos")

content=[] additional_kwargs={'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_1940889': 'EnEKbwERTTIPZlR4hJT2nccBcAWeYOiVbQk6+XqSzIEpRK2j0NNyKtP7zbr6jMYOJIq2nYYKDiEK8Dx7RjpNZeOtYk7DTq4D5nSyAkv5yDf8YvPWCWBEWgiPMhzEkqSUjQCyvVoGuDocVK1QktITP0gWWg=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a026e3-4606-7253-8788-5f2a1a4bd1da-0' tool_calls=[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_1940889', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 122, 'output_tokens': 23, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}}
Tempo: 1.94 segundos


Teste atual 3

In [7]:
from assistente.assistente import executar_agente
from assistente.llm import carregar_llm
from assistente.tools import consultar_cliente

In [6]:
#llm = carregar_llm()

#tools = [
#    consultar_cliente
#]

2026-08-21 21:17:30.887 | INFO     | assistente.llm:carregar_llm:26 - Inicializando LLM Gemini: gemini-3.1-flash-lite
2026-08-21 21:17:31.081 | INFO     | assistente.llm:carregar_llm:34 - LLM Gemini inicializada com sucesso.


In [9]:
resposta = executar_agente(
    pergunta="Consulte o cliente PJ0005 e informe o setor.",
    llm=llm,
    tools=tools
)

print(resposta.content)

2026-08-21 21:39:36.686 | INFO     | assistente.assistente:executar_agente:11 - Executando agente para pergunta: Consulte o cliente PJ0005 e informe o setor.
2026-08-21 21:39:41.360 | INFO     | assistente.assistente:executar_agente:28 - Tool Calls identificadas: 1
2026-08-21 21:39:41.361 | INFO     | assistente.assistente:executar_agente:50 - Executando tool: consultar_cliente
2026-08-21 21:39:41.363 | INFO     | assistente.tools:consultar_cliente:63 - Consultando cliente: PJ0005
2026-08-21 21:39:41.365 | INFO     | assistente.tools:carregar_clientes:26 - Carregando base estruturada de clientes.
2026-08-21 21:39:41.374 | INFO     | assistente.tools:carregar_clientes:30 - Base de clientes carregada com sucesso. Registros: 800
2026-08-21 21:39:41.377 | INFO     | assistente.tools:consultar_cliente:86 - Cliente PJ0005 encontrado com sucesso.
2026-08-21 21:39:41.378 | INFO     | assistente.assistente:executar_agente:67 - Tool consultar_cliente executada com sucesso.
2026-08-21 21:39:41.91

GoogleInvalidRequestError: Error calling model 'gemini-3.1-flash-lite' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Please ensure that function call turn comes immediately after a user turn or after a function response turn.', 'status': 'INVALID_ARGUMENT'}}

In [14]:
mensagem = llm_com_tools.invoke(
    "Consulte o cliente PJ0005 e informe o setor."
)

print(mensagem)

content=[] additional_kwargs={'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_1183112': 'EnEKbwERTTIPELFsbPHvpXwRlDyT9b3uBjJC7PHZeiwfN1Rt53hUWsopyFpauN/lSiag5DGhrAWtrkMIjzO22PowxEp+rmbhVIxSq3gDb5qhfwkWfHtxD8mZDBbIh3ftzRACiw0o8+AQyJHWUmhpopehOA=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a026d1-2446-7023-8243-f21fa16a0329-0' tool_calls=[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_1183112', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 124, 'output_tokens': 23, 'total_tokens': 147, 'input_token_details': {'cache_read': 0}}


In [15]:
resultado = consultar_cliente.invoke(
    {
        "id_cliente": "PJ0005"
    }
)

print(resultado)

2026-08-21 21:14:09.506 | INFO     | assistente.tools:consultar_cliente:63 - Consultando cliente: PJ0005
2026-08-21 21:14:09.507 | INFO     | assistente.tools:carregar_clientes:26 - Carregando base estruturada de clientes.
2026-08-21 21:14:09.522 | INFO     | assistente.tools:carregar_clientes:30 - Base de clientes carregada com sucesso. Registros: 800
2026-08-21 21:14:09.525 | INFO     | assistente.tools:consultar_cliente:86 - Cliente PJ0005 encontrado com sucesso.


{'sucesso': True, 'id_cliente': 'PJ0005', 'dados': {'id_cliente': 'PJ0005', 'data_referencia': '2026-02-28', 'setor': 'Comércio', 'porte': 'ME', 'uf': 'SC', 'tempo_relacionamento_meses': 208, 'faturamento_mensal_estimado': 96854.12, 'saldo_devedor': 19745.94, 'dias_atraso': 43, 'qtd_contratos_ativos': 2, 'qtd_parcelas_vencidas': 1, 'limite_credito': 83140.62, 'utilizacao_limite_pct': 0.1913, 'qtd_contatos_ult_30d': 4, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'whatsapp', 'risco_setorial': 'baixo', 'score_regularizacao_legado': 527, 'score_cobranca_atualizado_30d': 909, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 2, 'dia_referencia': 28, 'ano_mes_referencia': '2026-02', 'nome_mes_referencia': 'Fevereiro'}}


In [16]:
print(mensagem)
print(mensagem.content)
print(mensagem.tool_calls)
print(mensagem.additional_kwargs)

content=[] additional_kwargs={'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_1183112': 'EnEKbwERTTIPELFsbPHvpXwRlDyT9b3uBjJC7PHZeiwfN1Rt53hUWsopyFpauN/lSiag5DGhrAWtrkMIjzO22PowxEp+rmbhVIxSq3gDb5qhfwkWfHtxD8mZDBbIh3ftzRACiw0o8+AQyJHWUmhpopehOA=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a026d1-2446-7023-8243-f21fa16a0329-0' tool_calls=[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_1183112', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 124, 'output_tokens': 23, 'total_tokens': 147, 'input_token_details': {'cache_read': 0}}
[]
[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_1183112', 'type': 'tool_call'}]
{'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, 

In [ ]:
resposta = executar_agente(
    pergunta="Consulte o cliente PJ0005 e informe o setor.",
    llm=llm,
    tools=[consultar_cliente]
)

print(resposta.content)

2026-08-21 21:17:40.352 | INFO     | assistente.assistente:executar_agente:11 - Executando agente para pergunta: Consulte o cliente PJ0005 e informe o setor.


In [ ]:
mensagem = llm_com_tools.invoke(
    "Consulte o cliente com id PJ0005."
)

print(mensagem.tool_calls)

In [ ]:
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

chat = client.chats.create(
    model="gemini-3.1-flash-lite",
    config={
        "tools": [consultar_cliente]
    }
)

In [11]:
mensagem = llm_com_tools.invoke(
    "Consulte o cliente com id PJ0005."
)

print(mensagem.tool_calls)

[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_2096049', 'type': 'tool_call'}]


In [12]:
print(type(mensagem))
print(mensagem)

<class 'langchain_core.messages.ai.AIMessage'>
content=[] additional_kwargs={'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_2096049': 'EnEKbwERTTIPz0MOAh29YckfTgX0CZOWgbv6CfE2ChFhCDNGsQ9VXC7oXrkcIOqgktq78rbD/VNbSd0WQR1zzZokg0+644W7HlcBb8omPYCqcFb1Bux7QmNZWnhXgKwerKnBsO/91xZQX9W0GsYRU90+nw=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a026eb-21cd-79d3-9186-4e424ade9a83-0' tool_calls=[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_2096049', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 122, 'output_tokens': 23, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}}


In [13]:
print(mensagem.model_dump())

{'content': [], 'additional_kwargs': {'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_2096049': 'EnEKbwERTTIPz0MOAh29YckfTgX0CZOWgbv6CfE2ChFhCDNGsQ9VXC7oXrkcIOqgktq78rbD/VNbSd0WQR1zzZokg0+644W7HlcBb8omPYCqcFb1Bux7QmNZWnhXgKwerKnBsO/91xZQX9W0GsYRU90+nw=='}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--01a026eb-21cd-79d3-9186-4e424ade9a83-0', 'tool_calls': [{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_2096049', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 122, 'output_tokens': 23, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}}}


In [15]:
mensagem = llm_com_tools.invoke(
    "Consulte o cliente com id PJ0005."
)

print("Tipo:", type(mensagem))
print("Tool calls:", mensagem.tool_calls)

for tool_call in mensagem.tool_calls:
    print("Tool:", tool_call["name"])
    print("Args:", tool_call["args"])

Tipo: <class 'langchain_core.messages.ai.AIMessage'>
Tool calls: [{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_734897', 'type': 'tool_call'}]
Tool: consultar_cliente
Args: {'id_cliente': 'PJ0005'}


In [16]:
for tool_call in mensagem.tool_calls:
    if tool_call["name"] == "consultar_cliente":
        resultado = consultar_cliente.invoke(tool_call["args"])
        print(resultado)

2026-08-21 21:46:38.735 | INFO     | assistente.tools:consultar_cliente:63 - Consultando cliente: PJ0005
2026-08-21 21:46:38.751 | INFO     | assistente.tools:carregar_clientes:26 - Carregando base estruturada de clientes.
2026-08-21 21:46:38.930 | INFO     | assistente.tools:carregar_clientes:30 - Base de clientes carregada com sucesso. Registros: 800
2026-08-21 21:46:38.941 | INFO     | assistente.tools:consultar_cliente:86 - Cliente PJ0005 encontrado com sucesso.


{'sucesso': True, 'id_cliente': 'PJ0005', 'dados': {'id_cliente': 'PJ0005', 'data_referencia': '2026-02-28', 'setor': 'Comércio', 'porte': 'ME', 'uf': 'SC', 'tempo_relacionamento_meses': 208, 'faturamento_mensal_estimado': 96854.12, 'saldo_devedor': 19745.94, 'dias_atraso': 43, 'qtd_contratos_ativos': 2, 'qtd_parcelas_vencidas': 1, 'limite_credito': 83140.62, 'utilizacao_limite_pct': 0.1913, 'qtd_contatos_ult_30d': 4, 'promessa_pagamento_ult_30d': 0, 'renegociacoes_ult_12m': 0, 'canal_preferencial': 'whatsapp', 'risco_setorial': 'baixo', 'score_regularizacao_legado': 527, 'score_cobranca_atualizado_30d': 909, 'regularizou_30d': 1, 'ano_referencia': 2026, 'mes_referencia': 2, 'dia_referencia': 28, 'ano_mes_referencia': '2026-02', 'nome_mes_referencia': 'Fevereiro'}}


FLUXO FINAL 

In [9]:
from langchain_core.messages import HumanMessage, ToolMessage


pergunta = "Consulte o cliente com id PJ0005."

mensagem = llm_com_tools.invoke(
    [HumanMessage(content=pergunta)]
)

print("Primeira resposta:")
print(mensagem)
print()
print("Tool calls:")
print(mensagem.tool_calls)


mensagens = [
    HumanMessage(content=pergunta),
    mensagem
]


for tool_call in mensagem.tool_calls:

    if tool_call["name"] == "consultar_cliente":

        resultado = consultar_cliente.invoke(
            tool_call["args"]
        )

        print("\nResultado da tool:")
        print(resultado)

        mensagens.append(
            ToolMessage(
                content=str(resultado),
                tool_call_id=tool_call["id"]
            )
        )


resposta_final = llm_com_tools.invoke(mensagens)

print("\nResposta final:")
print(resposta_final.content)

2026-08-21 22:22:11.034 | INFO     | assistente.tools:consultar_cliente:63 - Consultando cliente: PJ0005
2026-08-21 22:22:11.035 | INFO     | assistente.tools:carregar_clientes:26 - Carregando base estruturada de clientes.
2026-08-21 22:22:11.052 | INFO     | assistente.tools:carregar_clientes:30 - Base de clientes carregada com sucesso. Registros: 800
2026-08-21 22:22:11.055 | INFO     | assistente.tools:consultar_cliente:86 - Cliente PJ0005 encontrado com sucesso.


Primeira resposta:
content=[] additional_kwargs={'function_call': {'name': 'consultar_cliente', 'arguments': '{"id_cliente": "PJ0005"}'}, '__gemini_function_call_thought_signatures__': {'call_2024161': 'EnEKbwERTTIPzr1r6qpJzpNnyuOx7LpxjWLEJumm6D6cvS0yijPoUdE/vXoTYN3i5VwLUJH6dRQ2Esz+TRZpLBHEQp1FrWgo38nka9V89L9FIcxBVRTN/hPJKVEpPqgNbkbmF0i90iHrO56jxSaQraJ4Ug=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0270f-9c75-72b1-903a-5e836990ef02-0' tool_calls=[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_2024161', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 122, 'output_tokens': 23, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}}

Tool calls:
[{'name': 'consultar_cliente', 'args': {'id_cliente': 'PJ0005'}, 'id': 'call_2024161', 'type': 'tool_call'}]

Resultado da tool:
{'sucesso': True, 'id_cliente': 'PJ0005',